# 07 · CUDA 커널 개념 & CuPy 고수준 커널

> **CuPy 2일 집중 코스 — Day 2 / 단원 5 (CuPy 고수준 커널 + 커널 개념, 1.0H)**

Day 2는 **사용자 정의 커널**입니다. 먼저 **CUDA 커널의 핵심 개념**을 충실히 잡고,
그 개념들이 **07→08→09→10→11** 노트북에서 각각 어떻게 구현되는지 지도를 그립니다.
그다음 가장 쉬운 시작인 CuPy의 `ElementwiseKernel`·`ReductionKernel`을 다룹니다.

## 학습 목표
- CUDA **실행 모델**(커널·thread/block/grid·SIMT/warp)을 설명한다.
- **메모리 계층**·**coalescing**·**동기화/atomic**·**occupancy** 개념을 이해한다.
- 이 개념들이 각 노트북(07~11)에서 어떻게 구현되는지 연결한다.
- `ElementwiseKernel`·`ReductionKernel`로 첫 커스텀 커널을 작성한다.

## 목차
### A. CUDA 커널 개념
1. [커널과 실행 모델](#1) · 2. [스레드 계층과 인덱싱](#2) · 3. [SIMT와 warp](#3)
4. [메모리 계층](#4) · 5. [메모리 접근(coalescing)](#5) · 6. [동기화와 atomic](#6) · 7. [occupancy](#7)
8. [개념 → 노트북 매핑](#8)
### B. CuPy 고수준 커널
9. [ElementwiseKernel](#9) · 10. [ReductionKernel](#10) · 11. [@cupy.fuse 관계](#11) · 12. [체크포인트](#12)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from numba import cuda
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. 커널과 실행 모델

**커널(kernel)** 은 *GPU에서 수천 개의 스레드로 동시에 실행되는 함수*입니다.
CPU(host)가 커널을 **런치(launch)** 하면 GPU가 지정한 수의 스레드를 만들어 같은 코드를 각자 다른 데이터에 적용합니다.
- 호스트가 `kernel[그리드, 블록](인자)` 로 실행 구성을 정해 런치
- 각 스레드는 **자신의 인덱스**로 처리할 데이터를 고름 (SPMD)
- 런치는 **비동기** (Day 1의 비동기와 동일)

> Day 1까지는 CuPy가 커널을 자동 생성했습니다. Day 2는 그 커널을 **직접** 들여다보고 작성합니다.

<a id="2"></a>
## 2. 스레드 계층과 인덱싱

| 계층 | 의미 | 공유 자원 |
|------|------|-----------|
| **thread** | 최소 실행 단위(보통 원소 1개) | 자신의 레지스터 |
| **block** | 스레드 묶음 | **공유 메모리**·`syncthreads` |
| **grid** | 블록 묶음(전체 문제) | 전역 메모리 |

**전역 인덱스**(1D): `global = blockIdx.x*blockDim.x + threadIdx.x`
- `threadIdx.x` 블록 내 위치 · `blockIdx.x` 블록 번호 · `blockDim.x` 블록당 스레드 수
- Numba는 `cuda.grid(1)` 이 위 식을 대신 계산(08). 
   * GPU 스레드의 전역 번호(인덱스)를 구하는 복잡한 수학 공식을 매번 쓸 필요 없이, Numba에서는 cuda.grid(1) 한 줄로 편하게 가져올 수 있음 
   * 뒤의 (1)은 1차원 배열을 의미하며, 2차원/3차원 배열은 cuda.grid(2)

<a id="3"></a>
## 3. SIMT와 warp

하드웨어는 스레드를 **warp(32개)** 단위로 묶어 **SIMT**(한 명령을 여러 스레드 동시)로 실행합니다.
- warp의 32 스레드는 **같은 명령을 lockstep**으로 수행
- warp 내 **분기(divergence)** → 경로별 직렬화로 성능 저하
- 많은 warp로 메모리 지연을 **겹쳐 숨김**(latency hiding)

> 시사점: **균일 연산·연속 접근**이 유리(08 coalescing, 분기 최소화).

### 3.1 CASE 1 (Warp 부족):

GPU 연산 코어(SM)에 일할 수 있는 Warp가 몇 개 없습니다. 메모리에서 A[i], B[i] 데이터를 가져오는 수백 클럭 동안 대신 일해줄 다른 Warp가 없어서 GPU 코어들이 멍하니 대기(Stall)하게 됩니다. 그 결과 데이터 처리 효율(Throughput)이 아주 좋지 않음



### 3.2 CASE 2 (Warp 풍족 - Latency Hiding):

GPU에 수십만 개의 Warp가 빽빽하게 줄을 서 있습니다. Warp 1번이 메모리를 읽어오느라 대기 상태에 빠지는 순간, 스케줄러가 0.1나노초만에 Warp 2번, Warp 3번으로 체인지하여 계산을 계속 돌립니다. 그 사이에 Warp 1번의 데이터가도착하므로, GPU 코어는 단 1클럭도 쉬지 않고 가동되어 전체 처리량이 상승.

In [ ]:
# 메모리 접근(지연)을 일으키는 아주 단순한 커널
@cuda.jit
def memory_heavy_kernel(A, B, C):
    i = cuda.grid(1)
    if i < A.size:
        # [지연 발생] VRAM(Global Memory)에서 데이터를 읽어옴 (수백 클럭 대기)
        a = A[i]
        b = B[i]
        
        # 간단한 수학 연산
        C[i] = a * b + 1.0

# 데이터 준비 (3천만 개 원소)
N = 30_000_000
A = cuda.to_device(np.ones(N, dtype=np.float32))
B = cuda.to_device(np.ones(N, dtype=np.float32))
C = cuda.device_array(N, dtype=np.float32)

threads_per_block = 256 # 블록당 8개의 Warp (256 / 32 = 8 Warps)

# -------------------------------------------------------------------
#  [CASE 1] Warp가 부족할 때 (Latency Hiding 실패)
# 일부러 데이터의 아주 작은 일부(32,000개 = Warp 10개 분량)만 처리하도록 스레드를 적게 던짐
# -------------------------------------------------------------------
blocks_few = 125  # 총 스레드: 125 * 256 = 32,000개 (Warp 1,000개 미만)

start = time.perf_counter()
memory_heavy_kernel[blocks_few, threads_per_block](A, B, C)
cuda.synchronize()
time_few = time.perf_counter() - start

# -------------------------------------------------------------------
#  [CASE 2] Warp가 아주 많을 때 (Latency Hiding 성공)
# 전체 데이터 3천만 개를 모두 처리하도록 대량의 스레드/Warp를 GPU에 던짐
# -------------------------------------------------------------------
blocks_many = (N + threads_per_block - 1) // threads_per_block # 총 스레드: 3,000만 개 (Warp 약 93만 개!)

start = time.perf_counter()
memory_heavy_kernel[blocks_many, threads_per_block](A, B, C)
cuda.synchronize()
time_many = time.perf_counter() - start

# -------------------------------------------------------------------
# 결과 비교 (초당 처리하는 데이터 개수: Throughput 계산)
# -------------------------------------------------------------------
throughput_few = (blocks_few * threads_per_block) / time_few / 1e6
throughput_many = N / time_many / 1e6

print(f"CASE 1 (Warp 부족) Throughput : {throughput_few:.2f} M-elements/sec")
print(f"CASE 2 (Warp 풍족) Throughput : {throughput_many:.2f} M-elements/sec")
print(f"=> Latency Hiding을 통해 초당 처리량이 약 {throughput_many / throughput_few:.1f}배 증가!")

<a id="4"></a>
## 4. 메모리 계층

| 메모리 | 범위 | 속도 | 용도 |
|--------|------|------|------|
| 레지스터 | 스레드 | 가장 빠름 | 지역 변수 |
| 공유(shared) | **블록** | 매우 빠름(온칩) | 블록 내 협력·재사용 |
| 전역(global) | 전체 | 느림(off-chip) | 입출력 배열 |
| 상수/텍스처 | 전체(읽기) | 캐시됨 | 읽기 전용 |

> 전략: **전역 접근↓**, 자주 쓰는 데이터를 **공유/레지스터**에 올려 재사용(09에서 구현).

<a id="5"></a>
## 5. 메모리 접근 — coalescing

전역 메모리는 **연속 주소를 한 트랜잭션**으로 읽을 때 가장 효율적입니다.
- **Coalesced(연속)**: warp 인접 스레드가 인접 주소 → 적은 트랜잭션 ✅
- **Strided/blocked(흩어짐)**: 멀리 떨어진 주소 → 트랜잭션 폭증 ⚠️

같은 일이라도 **접근 패턴**만으로 수 배 차이(08에서 직접 측정).

<a id="6"></a>
## 6. 동기화와 atomic

여러 스레드가 같은 메모리를 동시에 수정하면 **데이터 레이스**로 결과가 깨집니다.
- **`atomic`**: 읽기-수정-쓰기를 나눌 수 없는 한 번으로 (`cuda.atomic.add`)
- **`syncthreads()`**: 블록 내 스레드를 한 지점에서 동기화(공유메모리 사용 전후)

> 09 히스토그램: 레이스 → atomic → 공유메모리+atomic 으로 단계 수정.

<a id="7"></a>
## 7. occupancy(점유율)

**occupancy** = SM에 동시에 올라간 warp 비율.
- 높을수록 지연을 잘 숨김(무조건 빠른 건 아님)
- **블록당 스레드 수·레지스터·공유메모리** 사용량이 좌우
- 그래서 `threads_per_block`·`items_per_thread` **튜닝**이 중요(08·11에서 스윕).

<a id="8"></a>
## 8. 개념 → 노트북 매핑

| 개념 | 07 CuPy | 08 Numba copy | 09 Numba hist | 10 cccl | 11 RawKernel |
|------|--------|----------------|----------------|---------|--------------|
| 인덱싱 | 자동 | **직접** | 직접 | 추상화 | **직접(CUDA C)** |
| coalescing | 자동 | **측정** | 적용 | 내부 | 적용 |
| 공유메모리 | — | — | **사용** | 내부 | **사용(타일링/리덕션)** |
| atomic/동기화 | — | — | **사용** | 내부 | 사용 |
| occupancy 튜닝 | — | **스윕** | 적용 | 자동 | **스윕** |
| 난이도 | 가장 쉬움 | 중간 | 높음 | 낮음 | 높음(CUDA C) |

**07** 개념은 CuPy가 대신 처리(쉬움) · **08·09** Numba로 직접 · **10** 검증 알고리즘으로 추상화 · **11** CUDA C로 직접+개념 적용 기술.

<a id="9"></a>
## 9. ElementwiseKernel

**개념 적용**: '각 스레드가 원소 1개 처리'(2절)를 CuPy가 자동 인덱싱·런치. 우리는 **원소별 C 식**만 작성.
`ElementwiseKernel(in_params, out_params, operation, name)`.

In [ ]:
clamp_scale = cp.ElementwiseKernel(
    'float32 x, float32 lo, float32 hi, float32 scale', 'float32 y',
    '''
        float v = x;
        if (v < lo) v = lo;
        if (v > hi) v = hi;
        y = v * scale;
    ''', 'clamp_scale')
x = cp.random.standard_normal(10_000_000, dtype=cp.float32)
y = clamp_scale(x, cp.float32(-1), cp.float32(1), cp.float32(0.5))
print(float(y.mean()), float(y.std()))

**연습 — LeakyReLU**: `y = x>0 ? x : a*x` 를 ElementwiseKernel로 작성·검증.

In [ ]:
# TODO: leaky = cp.ElementwiseKernel('float32 x, float32 a','float32 y','...','leaky')
x_np = np.random.randn(2_000_000).astype(np.float32); a = np.float32(0.1)
ref = np.where(x_np>0, x_np, a*x_np)
# allclose(ref, leaky(cp.asarray(x_np), cp.float32(a)), name='leaky')

<details><summary>💡 해답 보기</summary>

```python
leaky = cp.ElementwiseKernel('float32 x, float32 a','float32 y',
                             'y = x > 0 ? x : a * x;','leaky_relu')
allclose(ref, leaky(cp.asarray(x_np), cp.float32(a)), name='leaky')
```
</details>

<a id="10"></a>
## 10. ReductionKernel

**개념 적용**: map→reduce 패턴(많은 스레드의 부분합을 합침)을 CuPy가 내부적으로 공유메모리·트리 리덕션으로 처리.
5요소: `in, out, map_expr, reduce_expr, post, identity, name`.

In [ ]:
l1 = cp.ReductionKernel('float32 x','float32 y',
    'fabsf(x)', 'a + b', 'y = a', '0.0f', 'l1_norm')
x_np = np.random.randn(2_000_000).astype(np.float32)
allclose(np.abs(x_np).sum().astype(np.float32), l1(cp.asarray(x_np)), rtol=1e-5, atol=1e-4, name='L1')

**연습 — 제곱합**: `map=x*x, reduce=a+b` 로 ReductionKernel 작성.

In [ ]:
# TODO: ss = cp.ReductionKernel('float32 x','float32 y','x*x','a+b','y=a','0.0f','sum_sq')
x_np = np.random.randn(2_000_000).astype(np.float32)
ref = (x_np**2).sum().astype(np.float32)
# allclose(ref, ss(cp.asarray(x_np)), rtol=1e-4, atol=1e-1, name='sum_sq')

<details><summary>💡 해답 보기</summary>

```python
ss = cp.ReductionKernel('float32 x','float32 y','x*x','a+b','y=a','0.0f','sum_sq')
allclose(ref, ss(cp.asarray(x_np)), rtol=1e-4, atol=1e-1, name='sum_sq')
```
</details>

<a id="11"></a>
## 11. `@cupy.fuse` 와의 관계

- 단순 원소/리덕션 융합이면 **`@cupy.fuse`**(03)가 가장 쉽습니다. C 식 제어가 필요하면 Elementwise/Reduction,
- 복잡한 인덱싱·공유메모리·atomic이 필요하면 → **Numba CUDA(08~09)**, CUDA C 직접 작성은 → **RawKernel(11)**.
- 평소 쓰던 일반 파이썬/CuPy 코드 위에 @cp.fuse() 데코레이터 딱 한 줄만 붙여주면 CuPy가 알아서 초고속 GPU 커널로 융합(Fusion)해 주는 가장 쉬운 최적화 도구

## 🧪 추가 연습 — 커널 더 다루기

**연습 A — 2입력 ElementwiseKernel**: `z = a*x + b*y` 를 작성하세요(입력 4개).

In [ ]:
# TODO: axpby = cp.ElementwiseKernel('float32 x, float32 y, float32 a, float32 b','float32 z','...','axpby')
x = cp.random.rand(1_000_000, dtype=cp.float32); y = cp.random.rand(1_000_000, dtype=cp.float32)
# allclose(2*cp.asnumpy(x)+3*cp.asnumpy(y), axpby(x,y,cp.float32(2),cp.float32(3)), name='axpby')

<details><summary>💡 해답 보기</summary>

```python
axpby = cp.ElementwiseKernel('float32 x, float32 y, float32 a, float32 b','float32 z',
                             'z = a*x + b*y;','axpby')
allclose(2*cp.asnumpy(x)+3*cp.asnumpy(y), axpby(x,y,cp.float32(2),cp.float32(3)), name='axpby')
```
</details>

**연습 B — `raw` 파라미터로 이웃 접근**: 
- ElementwiseKernel은 `raw`(임의 인덱싱)+루프 인덱스 `i`로 이웃 원소에 접근할 수 있습니다.
- 전진 차분 `y[i] = x[i+1]-x[i]`(마지막은 0)을 작성하세요.
- ElementwiseKerne에서 raw 키워드를 붙이면 CuPy가 자동으로 수행하는 1:1 인덱싱을 해제하고, C++ 스타일의 포인터 배열처럼 x[i+1]이나 x[i-1] 같은 임의의 인덱스(이웃 원소)에 자유롭게 접근할 수 있게 해줍니다. 이때 자동으로 제공되는 루프 변수 i는 현재 GPU 스레드가 처리 중인 원소의 인덱스 번호를 나타냅니다.

In [ ]:
# TODO: fwd = cp.ElementwiseKernel('raw float32 x, int32 n','float32 y',
#           'y = (i < n-1) ? x[i+1]-x[i] : 0.0f;', 'fwd_diff')
x = cp.arange(10, dtype=cp.float32); y = cp.empty(10, dtype=cp.float32)
# fwd(x, np.int32(x.size), y); print(cp.asnumpy(y))   # 1,1,...,0

<details><summary>💡 해답 보기</summary>

```python
fwd = cp.ElementwiseKernel('raw float32 x, int32 n','float32 y',
                           'y = (i < n-1) ? x[i+1]-x[i] : 0.0f;', 'fwd_diff')
x = cp.arange(10, dtype=cp.float32); y = cp.empty(10, dtype=cp.float32)
fwd(x, np.int32(x.size), y); print(cp.asnumpy(y))
# raw 입력은 브로드캐스팅으로 크기를 못 정하므로 출력 배열(y)로 루프 크기를 지정합니다.
```
</details>

**연습 C — 가중합 ReductionKernel**: `sum(x*w)` 를 ReductionKernel로 작성(입력 2개).

In [ ]:
# TODO: wsum = cp.ReductionKernel('float32 x, float32 w','float32 y','x*w','a+b','y=a','0.0f','wsum')
x = np.random.rand(1_000_000).astype(np.float32); w = np.random.rand(1_000_000).astype(np.float32)
# allclose((x*w).sum().astype('f4'), wsum(cp.asarray(x), cp.asarray(w)), rtol=1e-3, atol=1e-1, name='wsum')

<details><summary>💡 해답 보기</summary>

```python
wsum = cp.ReductionKernel('float32 x, float32 w','float32 y','x*w','a+b','y=a','0.0f','wsum')
allclose((x*w).sum().astype('f4'), wsum(cp.asarray(x), cp.asarray(w)), rtol=1e-3, atol=1e-1, name='wsum')
```
</details>

<a id="12"></a>
## 12. 체크포인트

- [ ] 커널·thread/block/grid·SIMT/warp를 설명할 수 있다
- [ ] 메모리 계층·coalescing·atomic/동기화·occupancy 개념을 안다
- [ ] 각 개념이 07~11에서 어떻게 구현되는지 매핑을 이해했다
- [ ] ElementwiseKernel·ReductionKernel로 커널을 작성했다

다음: **`08_numba_copy`** — Numba CUDA로 스레드 인덱싱과 **coalescing**을 직접 구현·측정합니다.